In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("../data/processed/features_ecomop_logs_1.csv")

df.head()

,success_vol,success_rt_avg,fail_vol,fail_rt_avg,hour,hour_sin,hour_cos,operation_add_to_cart,operation_browse_products,operation_checkout,operation_payment,operation_view_cart,total_vol,fail_ratio,latency_gap,vol_per_latency
0,121586,21.479,617,5.404,0,0.0,1.0,False,True,False,False,False,122203,0.005049,16.075,5436.318342
1,26345,30.134,436,10.757,0,0.0,1.0,False,False,False,False,True,26781,0.016280,19.377,860.185007
2,24172,47.051,585,31.103,0,0.0,1.0,True,False,False,False,False,24757,0.023629,15.948,515.223408
3,8301,179.964,275,107.498,0,0.0,1.0,False,False,True,False,False,8576,0.032062,72.466,47.390641
4,6288,203.375,294,174.204,0,0.0,1.0,False,False,False,True,False,6582,0.044660,29.171,32.205505


In [3]:
ALL_FEATURES = [
    "success_vol",
    "fail_vol",
    "success_rt_avg",
    "fail_rt_avg",
    "hour_sin",
    "hour_cos",
    "hour"
] + [c for c in df.columns if c.startswith("operation_")]

In [4]:
def get_features_for_target(target, features):
    return [f for f in features if f != target]

In [5]:
TARGETS = [
    "success_vol",
    "fail_vol",
    "success_rt_avg",
    "fail_rt_avg"
]

In [6]:
def train_model(X_train, y_train, X_val, y_val):
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        eval_metric="mae"
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    return model

In [7]:
models = {}
metrics = {}
feature_map = {}

for target in TARGETS:
    
    print(f"\nTraining model for: {target}")
    
    FEATURES = get_features_for_target(target, ALL_FEATURES)
    
    X = df[FEATURES]
    y = df[target]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = train_model(X_train, y_train, X_val, y_val)
    
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    
    models[target] = model
    feature_map[target] = FEATURES
    metrics[target] = mae
    print(f"{target} features: {len(FEATURES)}")
    print(f"{target} MAE: {mae:.4f}")


Training model for: success_vol
success_vol features: 11
success_vol MAE: 3448.3052

Training model for: fail_vol
fail_vol features: 11
fail_vol MAE: 59.6733

Training model for: success_rt_avg
success_rt_avg features: 11
success_rt_avg MAE: 27.4197

Training model for: fail_rt_avg
fail_rt_avg features: 11
fail_rt_avg MAE: 16.7841


In [8]:
import joblib
import os

bundle = {
    "models": models,
    "feature_map": feature_map,
    "targets": TARGETS
}

os.makedirs("models", exist_ok=True)

joblib.dump(bundle, "../models/v2/model_bundle.pkl")

print("Model saved successfully")

Model saved successfully
